# 01 — Data quality and EDA

This notebook verifies the public synthetic data contract before modeling. It intentionally works from the temporal train/validation/test files rather than recreating a random split.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
from telco_churn.data import load_splits
from telco_churn.features import add_active_product_count, split_xy
frames = load_splits(ROOT / 'data' / 'synthetic')


In [ ]:
summary = []
for name, df in frames.items():
    summary.append({
        'split': name, 'rows': len(df), 'columns': df.shape[1],
        'date_min': df['Snapshot Date'].min().date(),
        'date_max': df['Snapshot Date'].max().date(),
        'churn_rate': df['Churn Next 30D'].mean(),
        'unique_ids': df['Customer ID'].nunique(),
    })
pd.DataFrame(summary)

The partitions are chronological and customer IDs do not overlap. The target prevalence increases mildly over time by design, creating a simple out-of-time robustness check.

In [ ]:
combined = pd.concat([df.assign(split=name) for name, df in frames.items()], ignore_index=True)
assert combined['Customer ID'].nunique() == len(combined)
assert not combined.isna().any().any()
print('Data-contract checks passed.')

In [ ]:
by_date = combined.groupby('Snapshot Date', as_index=False)['Churn Next 30D'].mean()
ax = by_date.plot(x='Snapshot Date', y='Churn Next 30D', marker='o', legend=False, figsize=(8,4))
ax.set(title='Synthetic churn prevalence over time', ylabel='Churn rate', xlabel='Snapshot date')
plt.tight_layout(); plt.show()

In [ ]:
train = add_active_product_count(frames['train'])
key = train.groupby('Contract', as_index=False)['Churn Next 30D'].agg(['mean','count']).reset_index()
key

In [ ]:
sat = train.groupby('Preperiod Satisfaction Score')['Churn Next 30D'].mean()
sat.plot(kind='bar', figsize=(7,4), title='Churn by preperiod satisfaction — train')
plt.ylabel('Churn rate'); plt.tight_layout(); plt.show()

`Preperiod Satisfaction Score` is predictive but not deterministic. Its measurement timing is explicitly defined before the 30-day outcome window, unlike the ambiguous source field reviewed during the academic-project audit.